In [ ]:
import os 
import glob
import comet_ml
import torch
import torchaudio
from src.logger.utils import plot_spectrogram
from omegaconf import OmegaConf
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from IPython.display import Audio, display
import matplotlib.pyplot as plt

In [ ]:
ROOT = os.getcwd()

with initialize_config_dir(config_dir=os.path.join(ROOT, "src/configs"), version_base=None):
    cfg = compose(config_name="conformer_s")

wav_aug_cfgs = list(cfg.transforms.instance_transforms.train.audio.transforms)
spec_aug_cfgs = list(cfg.transforms.batch_transforms.train.spectrogram._args_)
get_spec = instantiate(cfg.transforms.instance_transforms.inference.get_spectrogram)

print([c._target_.split(".")[-1] for c in wav_aug_cfgs])
print([c._target_.split(".")[-1] for c in spec_aug_cfgs])

In [ ]:
AUDIO_PATH = sorted(glob.glob(os.path.join(ROOT, "data/datasets/librispeech/dev-clean/**/*.flac"), recursive=True))[0]
wav, sr = torchaudio.load(AUDIO_PATH)
wav = wav[:1]
print(wav.shape)

In [ ]:
spectrogram = get_spec(wav)
display(Audio(wav[0].numpy(), rate=16000))

In [ ]:
augmented = {}
for aug_cfg in wav_aug_cfgs:
    params = OmegaConf.to_container(aug_cfg, resolve=True)
    params["p"] = 1
    name = params["_target_"].split(".")[-1]
    out = instantiate(params)(wav.clone())

    augmented[name] = out
    print(name)
    display(Audio(out[0].numpy(), rate=16000))

In [ ]:
%matplotlib inline
n = 3
spec_augmented = {}
for aug_cfg in spec_aug_cfgs:
    name = aug_cfg._target_.split(".")[-1]
    aug = instantiate(aug_cfg)
    spec_augmented[name] = [aug(spectrogram.clone()) for _ in range(n)]

    fig, axes = plt.subplots(1, n + 1, figsize=(5 * (n + 1), 3))
    axes[0].pcolormesh(spectrogram[0])
    axes[0].set_title("original")
    for i, s in enumerate(spec_augmented[name]):
        axes[i + 1].pcolormesh(s[0])
        axes[i + 1].set_title(f"{name} #{i}")
    plt.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
ROOT = os.getcwd()

comet_ml.login()
conf = OmegaConf.load("src/configs/writer/cometml.yaml")
exp = comet_ml.Experiment(project_name=conf.project_name, workspace=conf.workspace)
exp.set_name("aug_check")

In [ ]:
def spec_image(spec, name):
    return plot_spectrogram(spec[0].detach().cpu(), name=name).permute(1, 2, 0).numpy()

def log_wav(name, signal):
    exp.log_audio(signal[0].numpy(), sample_rate=160000, file_name=f"{name}.wav")
    exp.log_image(spec_image(get_spec(signal), name), name=f"{name}_spec")

log_wav("original", wav)
for name, out in augmented.items():
    log_wav(name, out)

for name, examples in spec_augmented.items():
    for i, s in enumerate(examples):
        exp.log_image(spec_image(s, f"{name}_{i}"), name=f"{name}_{i}")

In [ ]:
exp.end()